In [108]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [109]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [110]:
def load_keywords(json_file="quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [111]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [112]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
                else:
                    print(f"Error extrayendo ejercicios de archivo: {file.name}, página {idx+1}")
                    with open("errores.txt", "a") as f:
                        f.write(texto)
                        
    return exercises

In [113]:
def classify_exercise(statement: str, topic: str, keywords = None) -> tuple:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    unique_topic_keywords = list(keywords[topic]["keywords_tema"])
    for key, value in keywords[topic]["tipos_ejercicio"].items():
        unique_topic_keywords.extend(value)

    unique_topic_keywords = set(unique_topic_keywords)


    
    pattern = re.compile("|".join(unique_topic_keywords), flags=re.IGNORECASE | re.DOTALL)
    if re.search(pattern, statement):
        exercise_topic = topic
    else:
        exercise_topic = None

    if exercise_topic:
        exercise_types = []
        exercise_type_keywords = keywords[topic]["tipos_ejercicio"]
        
        for tipo, tipo_keywords in exercise_type_keywords.items():
            pattern = re.compile("|".join(tipo_keywords), flags=re.IGNORECASE | re.DOTALL)
            if re.search(pattern, statement):
                exercise_types.append(tipo)
    else:
        exercise_types = [] 

    return exercise_topic, exercise_types

In [114]:
def create_df_exercises(folder: Path) -> pd.DataFrame:
    exercises_list = []
    if not isinstance(folder, Path):
        raise ValueError("La función espera un objeto Path con el directorio a procesar.")
        if not folder.is_dir():
            raise ValueError("El objeto Path debe ser un directorio.")
    
    for file in folder.iterdir():
        if not file.suffix == ".pdf": 
            continue
        exams, statements = [], []
        #print("Processing file: ", file.name)
        exercises = extract_exercises(file)
        for ex in exercises:
            if ex:
                exercises_list.append({"exam" : ex[0], "statement": ex[1]})
    return pd.DataFrame(exercises_list)

Clasificar ejercicios de una carpeta

In [115]:
# # RENOMBRAR CARPETAS PARA QUE COINCIDAN CON CLAVES DEL JSON CON KEYWORDS
# import shutil

# acentos = {"á" : 'a', "é" : 'e', 'í' : 'i', 'ó' : 'o'}
# translation_table = str.maketrans(acentos)

# for folder in Path("temas").iterdir():
#     new_folder = folder.name.lower().replace(" ", "_")
#     new_folder = new_folder.translate(translation_table)
    
#     new_folder = Path("/".join(folder.parts[:-1]) + f"/{new_folder}")
#     shutil.move(folder, new_folder)

Ordenar directorios sin acentos:

In [116]:
TEMAS = sorted(Path("temas").iterdir())

In [117]:
for idx, topic in enumerate([i.name for i in TEMAS]):
    print(f"{idx}: {topic}")

0: acido_base
1: cantidad_quimica
2: configuracion_electronica
3: enlace_quimico
4: equilibrio_quimico
5: formulacion
6: reactividad_organica
7: redox
8: solubilidad
9: termoquimica


## Clasificar carpeta de ejercicios

In [151]:
df = create_df_exercises(TEMAS[9])


In [174]:
df["topic"], df["exercise_type"] = zip(*df.statement.apply(classify_exercise, topic=TEMAS[9].name))
df = df.explode("exercise_type")
df = df.sort_values("exam").reset_index(drop=True)
df = df.drop_duplicates()

In [175]:
df.isna().sum()

exam             0
statement        0
topic            3
exercise_type    3
dtype: int64

In [176]:
df[df.topic.isna()]

,exam,statement,topic,exercise_type
786,QUÍMICA.2025. JULIO. EJERCICIO 5,,NaN,NaN
787,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,,NaN,NaN
788,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,QUIMICA. 2026. RESERVA 4. EJERCICIO 5,NaN,NaN


In [177]:
for i in df[df.exercise_type.isna()].statement:
    print(f"{i}\n\n")







QUIMICA. 2026. RESERVA 4. EJERCICIO 5




In [146]:
for i in df[df.topic.isna()].statement.head():
    print(f"{i}\n\n")

In [61]:
df_cinetica = df[df.topic.isna()]

In [62]:
df.exercise_type.unique()

<StringArray>
[     'Principio de Le Chatelier',             'Cálculo de Kc y Kp',
 'Grado de disociación (cálculo)',              'verdadero o falso',
                  'Ley de Dalton',         'Equilibrio heterogéneo',
  'Equilibrio en función de alfa',       'Grado disociación (dato)',
                              nan,           'Cociente de reacción']
Length: 10, dtype: str

In [63]:
df_cinetica["topic"], df_cinetica["exercise_type"] = zip(*df_cinetica.statement.apply(classify_exercise, topic="cinetica"))
df_cinetica = df_cinetica.explode("exercise_type")
df_cinetica = df_cinetica.sort_values("exam").reset_index(drop=True)
df_cinetica = df_cinetica.drop_duplicates()

In [64]:
df_cinetica

,exam,statement,topic,exercise_type
0,QUÍMICA. 2000. RESERVA 1. EJERCICIO 4. OPCIÓN B,Dada la reacción: CO(g)+NO₂(g) CO₂(g)+NO(g)\na...,cinetica,NaN
1,QUÍMICA. 2004. JUNIO. EJERCICIO 3. OPCIÓN B,Se ha comprobado experimentalmente que la reac...,cinetica,cinética
2,QUÍMICA. 2006. SEPTIEMBRE. EJERCICIO 3. OPCIÓN B,La reacción: A+2B → 2C+Des de primer orden con...,cinetica,cinética
3,QUÍMICA. 2008. SEPTIEMBRE. EJERCICI O 3. OPCIÓN B,"A una hipotética reacción química, A + B → C, ...",cinetica,cinética
4,QUÍMICA. 2009. JUNIO. EJERCICIO 3. OPCIÓN B,Para el proceso: 2NO(g) + 2H₂(g) → N (g) + 2H₂...,cinetica,cinética
5,QUÍMICA. 2017. SEPTIEMBRE. EJERCICIO 4. OPCIÓN B,"La reacción: A + 2B + C → D + E, tiene como ec...",cinetica,cinética
6,QUÍMICA. 2020. SEPTIEMBRE. B6,"La reacción: A+2B → C, es de orden cero con re...",cinetica,cinética
7,QUÍMICA. 2022. JUNIO. EJERCICIO B2,La reacción: A + B → C + D es de primer orden ...,cinetica,cinética
8,QUÍMICA. 2023. RESERVA 1. EJERCICIO B6,"La reacción: X+2Y → M, es de orden dos respect...",cinetica,cinética
9,QUÍMICA. 2024. RESERVA 3. EJERCICIO B6,La reacción A + B → C es de orden cero con res...,cinetica,cinética


In [65]:
for i in df_cinetica[df_cinetica.topic.isna()].statement:
    print(f"{i}\n\n")